# ローカルで完結する対話アプリの作成

## 参考

- [Reachy Mini goes fully local (May 27, 2026, Hugging Face)](https://huggingface.co/blog/local-reachy-mini-conversation)
    - [huggingface/speech-to-speech](https://github.com/huggingface/speech-to-speech)
- [NVIDIA brings agents to life with DGX Spark and Reachy Mini (Jan 5 2026, Hugging Face)](https://huggingface.co/blog/nvidia-reachy-mini)
    - [brevdev/reachy-personal-assistant](https://github.com/brevdev/reachy-personal-assistant)
- [Building a Fully Local Voice AI Agent on a Reachy Mini Robot (April 6 2026, Hugging Face)](https://huggingface.co/blog/curtburk/reachy-voice-agent)

## 概要

huggingface/speech-to-speechを参考にして、ローカル音声エージェントを作成する。このリポジトリは以下のコンポーネントから構成されるSpeech to Speechのカスケード型パイプラインを採用している。

1. VAD（Voice Activity Detection, 音声区間検出）
1. STT （Speech to Text, 音声認識）
1. LM（Language Model, 大規模言語モデル）
1. TTS（Text to Speech, 音声合成）

## ローカルで完結させるメリット

ホスト型のリアルタイム・バックエンドは便利だが、独自のエンジンを運用することで以下のメリットが得られる。

| メリット | 説明 |
|----------|------|
| **プライバシー** | 音声データがネットワークの外に出ることがなく、パイプライン全体を自身が管理するハードウェア上で実行できる。 |
| **APIコストゼロ** | 従量課金やトークンごとの費用が発生しない。 |
| **パイプラインの完全な制御** | VAD・STT・LLM・TTS のすべてのパーツを入れ替え可能。Hugging Face Hu にさらに優れたモデルが登場した際、いつでも変更できる。 |
| **デプロイの信頼性** | クラウド依存のデモはネットワーク環境で失敗することがある。ホテルのWi-FiはAPIコールを制限したり、企業ネットワークは外部の推論エンドポイントへの通信を遮断することがある。ローカルハードウェアで動作する隔離されたシステムは、ネットワーク環境に左右されず安定して動く。 |

## 仮想環境を構築

Jetson CUDA 12.6向けのPyTorchホイールはPython 3.10（cp310）用なので、このノートブックではPython 3.10の仮想環境を使う。既存のPython 3.12環境ではJetson CUDA 12.6用のPyTorchホイールが入らないため、ここでは再利用しない。

```sh
python3.10 -m venv .reachy_mini_local_conversation_cuda126
source .reachy_mini_local_conversation_cuda126/bin/activate
python -m pip install -U pip setuptools wheel ipykernel
python -m ipykernel install --user --name reachy-mini-local-conversation-cuda126 --display-name "Reachy Mini Local Conversation (Python 3.10 CUDA 12.6)"
```

仮想環境を作成したら、Jupyterのカーネルを`Reachy Mini Local Conversation (Python 3.10 CUDA 12.6)`に切り替えてから以降のセルを実行する。

## Speech To Speechを動かす

In [ ]:
import os
import sys

assert sys.version_info[:2] == (3, 10), sys.version
print(sys.executable)
print(sys.version)

if not os.path.exists("speech-to-speech"):
    !git clone https://github.com/huggingface/speech-to-speech.git

### Kokoro TTSを使う理由

このワークショップ環境はLinux ARM64（aarch64）で動作しているため、Qwen3 TTSのGGMLバックエンドが依存する`qwentts-cpp-python`の対応ホイールが見つからず、`speech-to-speech`のインストールが失敗することがある。そのため、このノートブックではQwen3ではなく、CPUでも導入しやすい`Kokoro-82M`ベースのKokoro TTSを使用する。対応するCUDA環境などでQwen3を使いたい場合は、`speech-to-speech[qwen3]`を明示的にインストールして`--tts qwen3`を指定する。

In [ ]:
%pip install -U pip setuptools wheel filelock typing-extensions sympy networkx jinja2 fsspec numpy==1.26.4
%pip install --no-cache-dir --no-deps --index-url https://pypi.jetson-ai-lab.io/jp6/cu126/+simple torch==2.8.0 torchaudio==2.8.0

import torch
import numpy as np

print(np.__version__)
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no cuda")
assert int(np.__version__.split(".", 1)[0]) < 2, "Use numpy==1.26.4 with the Jetson PyTorch wheel."
assert torch.cuda.is_available(), "PyTorch CUDA is not available. Check that this notebook uses the Python 3.10 CUDA 12.6 kernel."

%pip install -e "speech-to-speech[kokoro]"

In [ ]:
%pip show speech-to-speech

In [ ]:
%pip install python-dotenv

from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY is not None, "OPENAI_API_KEY is not set in the environment variables."

In [ ]:
import sys

!{sys.executable} -m speech_to_speech.s2s_pipeline \
  --stt whisper \
  --stt_model_name openai/whisper-large-v3-turbo \
  --stt_device cuda \
  --stt_torch_dtype float16 \
  --language ja \
  --llm_backend responses-api \
  --model_name gpt-4o-mini \
  --tts kokoro \
  --kokoro_device cpu \
  --kokoro_lang_code j \
  --kokoro_voice jf_alpha \
  --mode realtime \
  --enable_live_transcription


In [ ]:
import sys

!{sys.executable} -m speech_to_speech.s2s_pipeline \
  --stt whisper \
  --stt_model_name openai/whisper-large-v3-turbo \
  --stt_device cuda \
  --stt_torch_dtype float16 \
  --language ja \
  --llm_backend responses-api \
  --model_name gpt-4o-mini \
  --tts kokoro \
  --kokoro_device cpu \
  --kokoro_lang_code j \
  --kokoro_voice jf_alpha \
  --mode realtime

In [ ]:
import sys

!{sys.executable} -m speech_to_speech.s2s_pipeline --tts kokoro --kokoro_device cpu --kokoro_lang_code j --kokoro_voice jf_alpha

In [ ]:
# Mac / Apple Silicon only:
# !speech-to-speech --local_mac_optimal_settings

Windows


```sh
# Qwen3.5
llama-server.exe -hf unsloth/Qwen3.5-0.8B-GGUF
llama-server.exe -hf unsloth/Qwen3.5-2B-GGUF
llama-server.exe -hf unsloth/Qwen3.5-4B-GGUF


# Gemma 4
llama-server.exe -hf ggml-org/gemma-4-E2B-it-GGUF
```


```sh
python -m speech_to_speech.s2s_pipeline `
  --stt whisper `
  --stt_model_name openai/whisper-large-v3-turbo `
  --stt_device cuda `
  --stt_torch_dtype float16 `
  --language ja `
  --llm_backend responses-api `
  --model_name gemma-4-E2B-it `
  --responses_api_base_url http://127.0.0.1:8080/v1 `
  --tts kokoro `
  --kokoro_device cpu `
  --kokoro_lang_code j `
  --kokoro_voice jf_alpha `
  --mode realtime `
  --enable_live_transcription
```

```sh
python -m speech_to_speech.s2s_pipeline --stt whisper --stt_model_name openai/whisper-large-v3-turbo --stt_device cuda --stt_torch_dtype float16 --language ja --llm_backend responses-api --model_name gemma-4-E2B-it --responses_api_base_url http://127.0.0.1:8080/v1 --responses_api_api_key dummy --tts qwen3 --qwen3_tts_device cuda --qwen3_tts_language ja --mode realtime --enable_live_transcription
```

python -m speech_to_speech.s2s_pipeline --stt whisper --stt_model_name openai/whisper-large-v3-turbo --stt_device cuda --stt_torch_dtype float16 --language ja --llm_backend responses-api --model_name gemma-4-E2B-it --responses_api_base_url http://127.0.0.1:8080/v1 --responses_api_api_key dummy --tts qwen3 --qwen3_tts_device cuda --qwen3_tts_language japanese --mode realtime --enable_live_transcription


python -m speech_to_speech.s2s_pipeline --stt whisper --stt_model_name openai/whisper-large-v3-turbo --stt_device cuda --stt_torch_dtype float16 --language ja --llm_backend responses-api --model_name qwen3.5-0.8B-GGUF --responses_api_base_url http://127.0.0.1:8080/v1 --responses_api_api_key dummy --tts qwen3 --qwen3_tts_device cuda --qwen3_tts_language japanese --mode realtime --enable_live_transcription